In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadMergeTime\BoSSSpad.dll"
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadMerge\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XdgTimestepping;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
using BoSSS.Foundation.Grid.Classic;
using ilPSP.Utils;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;
using BoSSS.Foundation.Grid;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.Control;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
using BoSSS.Solution.XNSECommon;

In [3]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client @C:\Users\miao\Documents\Database
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC3, @\\dc3\userspace\miao\cluster"
2,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC3, @\\dc3\userspace\miao\cluster"


In [4]:
GetDefaultQueue()

DeploymentBaseDirectory,C:\Users\miao\Documents\Database
DeployRuntime,False
RuntimeLocation,<null>
Name,<null>
DotnetRuntime,dotnet
BatchInstructionDir,<null>
AllowedDatabasesPaths,"[ C:\Users\miao\Documents\Database, C:\ ]"


In [5]:
static var myBatch = ExecutionQueues[0];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("EE-Beam-3D");

In [6]:
BoSSSshell.WorkflowMgm.Init("EE-Beam-3D");

Project name is set to 'EE-Beam-3D'.
Default Execution queue is chosen for the database.
Opening existing database 'C:\Users\miao\Documents\Database\EE-Beam-3D'.


In [7]:
wmg.DefaultDatabase

{ Session Count = 38; Grid Count = 94; Path = C:\Users\miao\Documents\Database\EE-Beam-3D }

## Create grid

In [8]:
public static class GridFactory {
 
    public static Grid3D GenerateGrid() { 
        double xLeft = -3;
        double xRight = 8;
        double ySize = 2;
        double zSize = 1;

        int kelem = 4;

        double[] Xnodes = GenericBlas.Linspace(xLeft, xRight, ((int)(xRight - xLeft) * kelem) + 1);
        double[] Ynodes = GenericBlas.Linspace(0, ySize, (int)(ySize) * kelem + 1);
        double[] Znodes = GenericBlas.Linspace(-0.5 * zSize / 2, 0.5 * zSize / 2, (int)(zSize) * kelem / 2 + 1);
        var grd = Grid3D.Cartesian3DGrid(Xnodes, Ynodes, Znodes);
                                        //periodicX: false, periodicY: false, periodicZ: false);

        grd.EdgeTagNames.Add(1, "wall_lower");
        //grd.EdgeTagNames.Add(1, "freeslip_lower");
        grd.EdgeTagNames.Add(2, "freeslip_upper");
        grd.EdgeTagNames.Add(3, "velocity_inlet_left");
        grd.EdgeTagNames.Add(4, "pressure_outlet_right");
        grd.EdgeTagNames.Add(5, "freeslip_front");
        grd.EdgeTagNames.Add(6, "freeslip_back");        

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;

            if(Math.Abs(X[1] - 0) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - ySize) <= 1.0e-8)
                et = 2;
            if(Math.Abs(X[0] - xLeft) <= 1.0e-8)
                et = 3;
            if(Math.Abs(X[0] - xRight) <= 1.0e-8)
                et = 4;
            if(Math.Abs(X[2] - 0.5 * zSize / 2) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[2] + 0.5 * zSize / 2) <= 1.0e-8)
                et = 1;

            return et;
        });

        return grd;
     }
 
 }

In [9]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double InletVelocity(double[] X, double t) {");
           stw.WriteLine("    double R(double t) {");
           stw.WriteLine("        if(t <= 1) {");
           stw.WriteLine("            return (35 + (-84 + (70 - 20 * t) * t) * t) * t * t * t * t;");
           stw.WriteLine("        } else {");
           stw.WriteLine("            return 1;");
           stw.WriteLine("        }");
           stw.WriteLine("    }");
           stw.WriteLine("    double d = 0.2;");
           stw.WriteLine("    double G(double[] X) {");
           stw.WriteLine("        if(X[1] >= d)");
           stw.WriteLine("            return 1;");
           stw.WriteLine("        else");
           stw.WriteLine("            return (X[1] / d) * (X[1] / d);");
           stw.WriteLine("    }");
           stw.WriteLine("    double vmax = 1;");
           stw.WriteLine("    return vmax * R(t) * G(X);");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return -1;");
           stw.WriteLine("    }"); 
           
           //stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           //stw.WriteLine("    if(X[1] >= 1)");
           //stw.WriteLine("    return -((X[0]).Pow2() + (X[2]).Pow2() + (X[1] - 1.0).Pow2()) + 0.01;");
           //stw.WriteLine("        else");
           //stw.WriteLine("    return -((X[0]).Pow2() + (X[2]).Pow2()) + 0.01;");
           //stw.WriteLine("    }"); 

           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    double h = 0.1;");
           stw.WriteLine("    if(X[0] < 0 && X[1] < (1 - h)) {");
           stw.WriteLine("        return -(X[0]).Pow2() + h * h;");
           stw.WriteLine("    }");
           stw.WriteLine("    if(X[0] > 0 && X[1] < (1 - h)) {");
           stw.WriteLine("        return -(X[0]).Pow2() + h * h;");
           stw.WriteLine("    }");
           stw.WriteLine("    return -((X[0]).Pow2() + (X[1] - (1 - h)).Pow2()) + h * h;");
           //stw.WriteLine("    return -((X[0]).Pow2() + (X[1] - 0.4).Pow2()) + 0.25;");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InletVelocity(){
        return new Formula("BoundaryValues.InletVelocity", true, AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [10]:
public static ZLS_Control Channel( int p = 2, int AMRlvl = 2, double BeamDensity = 1) {
    ZLS_Control C = new ZLS_Control(p);
    C.AgglomerationThreshold = 0.2;
    C.NoOfMultigridLevels = 1;
    int D = 3;
    AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

    #region db
    C.savetodb = true;
    C.ProjectName = "Beam3D";
    C.SessionName = "Beam3D-Density" + BeamDensity + "-in-cross-flow";
    //C.ProjectDescription = "Droplet running on pc";
    C.ContinueOnIoError = false;
    //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
    //C.PostprocessingModules.Add(new MovingContactLineLogging());

    if(D == 3) {
        C.FieldOptions.Add("DisplacementZ", new FieldOpts() {
            Degree = p,
            SaveToDB = FieldOpts.SaveToDBOpt.TRUE
        });
    }

    #endregion
    
    // Physical Parameters
    // ===================
    #region physics
    C.PhysicalParameters.rho_A = 1.0;
    C.PhysicalParameters.rho_B = 1.0;
    //C.PhysicalParameters.mu_A = 0.05;
    //C.PhysicalParameters.mu_B = 0.05;
    C.PhysicalParameters.mu_A = 2.0;
    C.PhysicalParameters.mu_B = 2.0;
    //double sigma = 0.046;
    //C.PhysicalParameters.Sigma = sigma;
    //C.PhysicalParameters.betaS_A = 0.0;
    //C.PhysicalParameters.betaS_B = 0.0;
    //C.PhysicalParameters.betaL = 0.0;
    //C.PhysicalParameters.theta_e = Math.PI / 2.0;
    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = false; //??
    C.Material = new Solid() {
        Density = BeamDensity,
        Lame2 = 1e3,
        //Viscosity = 0.1
        Viscosity = 1.0
    };
    #endregion
    // grid generation
    // ===============
    #region grid
    C.SetGrid(GridFactory.GenerateGrid());
    #endregion
    // Initial Values
    // ==============
    //
    C.AddInitialValue("VelocityX#A", BoundaryValueFactory.Get_Zero());
    C.AddInitialValue("VelocityX#B", BoundaryValueFactory.Get_Zero());
    C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
    C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
    //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());
    // boundary conditions
    // ===================
    #region BC
    C.AddBoundaryValue("wall_lower");
    C.AddBoundaryValue("freeslip_upper");
    //C.AddBoundaryValue("freeslip_front");
    //C.AddBoundaryValue("freeslip_back");
    C.AddBoundaryValue("velocity_inlet_left", "VelocityX#A", BoundaryValueFactory.Get_InletVelocity());
    //C.AddBoundaryValue("velocity_inlet_left", "VelocityX#B", BoundaryValueFactory.Get_InletVelocity());
    C.AddBoundaryValue("pressure_outlet_right");
    C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
    C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
    //C.PhysicalParameters.sliplength = 0.001;
    #endregion
    // misc. solver options
    // ====================
    #region solver
    //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
    //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
    //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;
    C.NonLinearSolver.MaxSolverIterations = 80;
    C.NonLinearSolver.MinSolverIterations = 1;
    //C.Solver_MaxIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    //C.Solver_ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-12;
    //C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;
    //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
    //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
    //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
    //C.fullReInit = false; 
    C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
    C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.Standard;
    C.AdaptiveMeshRefinement = true;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband { maxRefinementLevel = AMRlvl });
    //C.activeAMRlevelIndicators.Add(new AMRatInnerContactLine { maxRefinementLevel = 3, levelSets = new[] { 1 }, FieldWidth = 1 });
    C.AMR_startUpSweeps = AMRlvl;
    #endregion

    C.DynamicLoadBalancing_On = true;
    C.DynamicLoadBalancing_Period = 50;
    C.DynamicLoadBalancing_RedistributeAtStartup = true;

    //C.GridPartType = GridPartType.clusterHilbert;
    C.GridPartType = GridPartType.METIS;

    // Timestepping
    // ============
    #region time
    //C.CheckJumpConditions = true;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Newton;
    
    C.TimesteppingMode = compMode;
    double dt = 1e-2;
    C.dtMax = dt;
    C.dtMin = dt;
    //C.Endtime = 100;
    C.NoOfTimesteps = 1000;
    C.saveperiod = 1;
    #endregion
    return C;
}

## Send and run jobs

In [11]:
//double[] Density = new double[] {0.1, 1, 10, 100}; 
double[] Density = new double[] {1}; 

In [12]:
foreach(double Den in Density){
    var C_CTRLFILE = Channel(2, 2, Den);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = 4;
    JobLocal.Activate();
    JobLocal.ShowOutput(); 
}

Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Beam3D-Density1-in-cross-flow ... 
Opening existing database '\\dc3\userspace\miao\cluster\EE-Beam-3D'.
Set Database: { Session Count = 37; Grid Count = 90; Path = C:\Users\miao\Documents\Database\EE-Beam-3D }
Grid is not in database yet...
Grid successfully saved: a06213fc-e2f1-4e92-8447-5627305941d7


Deploying executables and additional files ...
Deployment directory: C:\Users\miao\Documents\Database\EE-Beam-3D-ZwoLevelSetSolver2024Mar13_001453.321225
copied 46 files.
   written file: control.obj
deployment finished.
Starting mini batch processor in external process...
Started mini batch processor on local machine, process id is 15668.
started.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [13]:
databases[0].Sessions

#0: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	3/12/2024 11:52:28 PM	72090cd0...
#1: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	3/12/2024 11:14:27 PM	8b407071...
#2: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	3/12/2024 11:12:37 PM	bc6face1...
#3: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	3/12/2024 11:09:14 PM	f405e2bb...
#4: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	3/12/2024 11:05:47 PM	3521623a...
#5: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	3/12/2024 10:53:05 PM	a2de1f34...
#6: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	3/12/2024 10:51:04 PM	80bf5a5e...
#7: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	2/22/2024 8:30:36 PM	58ecb5ca...
#8: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	2/22/2024 5:24:29 PM	64822991...
#9: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	2/22/2024 10:01:46 AM	c65a8957...
#10: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	2/22/2024 9:56:39 AM	5d54e565...
#11: EE-Beam-3D	Beam3D-Density1-in-cross-flow*	2/21/2024 4:59:33 PM	66f72c9a...
#12: EE-Beam-3D	Beam3D-Density1-in-cross-f

In [13]:
databases[0].Sessions[0].Timesteps.Count

5

In [15]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(5).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\EE-Beam-3D__Beam3D-Density1-in-cross-flow__8d19e3ee-406c-47df-82fc-6ee8477298f8


## Post processing

In [ ]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [ ]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

In [ ]:
databases[0].Sessions

In [ ]:
var session = databases[0].Sessions[0];

In [ ]:
session.Timesteps.Count

In [ ]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("VelocityY").ProbeAt(0.405, 0),
        t => "VelocityY"
        );

In [ ]:
mydataset.PlotNow()

In [ ]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(518).Export().WithSupersampling(3).Do();

In [ ]:
var DispY = session.Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementY").Single();

In [ ]:
DispY

In [ ]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("Phi2"),
        t => "DisplacementY"
        );

In [ ]:
mydataset.PlotNow()

## Restart

In [ ]:
databases[0].Sessions

In [ ]:
//var Sloc = databases[0].Sessions.Last();
var Sloc = databases[0].Sessions[0];
Sloc

In [ ]:
var c2 = Sloc.CreateRestartControl() as ZLS_Control;

In [ ]:
c2.GetType()

In [ ]:
c2.GridGuid

In [ ]:
Sloc.Timesteps.Last().GridID

In [ ]:
c2.GridGuid = Sloc.Timesteps.Last().GridID;

In [ ]:
c2.GridGuid

In [ ]:
//c2.DynamicLoadBalancing_On = true;
//c2.DynamicLoadBalancing_Period = 10;
c2.DynamicLoadBalancing_RedistributeAtStartup = false;
//c2.GridPartType = GridPartType.METIS;

//c2.AdaptiveMeshRefinement = true;
//c2.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 5});
//c2.AMR_startUpSweeps = 5;
c2.AdaptiveMeshRefinement = false;

In [ ]:
//c2.TimeSteppingScheme = TimeSteppingScheme.BDF2;

In [ ]:
var JobLocal2 = c2.CreateJob();
JobLocal2.Status

In [ ]:
JobLocal2.NumberOfMPIProcs = 2;
JobLocal2.Activate();
JobLocal2.ShowOutput();

In [ ]:
databases[0].Sessions

In [ ]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

In [ ]:
databases[0].Sessions[0].